# Train Sentiment Overall Model bằng PhoBERT

Notebook này dùng để train model đánh giá **cảm xúc tổng thể của cuộc hội thoại** giữa:

- CSKH và khách hàng
- Chatbot và khách hàng

Pipeline gồm:

1. Load dữ liệu sentiment từ `data/raw/main_train/01_sentiment_overall`
2. Tiền xử lý dữ liệu
3. Chuẩn hóa nhãn sentiment
4. Tokenize bằng PhoBERT tokenizer
5. Build model PhoBERT classification
6. Train model
7. Validation/Test
8. Nhập thử hội thoại và dự đoán cảm xúc

Output model sẽ được lưu vào:

```text
models/sentiment_phobert/final_model/
```

## 1. Cài thư viện

Chạy cell này trước.

Nếu bạn chạy local và đã cài thư viện rồi thì có thể bỏ qua.

In [1]:
!pip install pandas scikit-learn transformers datasets accelerate underthesea -q

## 2. Import thư viện và cấu hình

Bạn có thể chỉnh các biến như `DATA_DIR`, `MODEL_NAME`, `NUM_EPOCHS`, `BATCH_SIZE`.

In [2]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

d:\learning\CareScore-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.11.0+cu128
CUDA available: True


In [3]:
# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path("data/raw/main_train/01_sentiment_overall")
OUTPUT_DIR = Path("models/sentiment_phobert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# PhoBERT phù hợp cho tiếng Việt.
# Nếu bị lỗi tải model, có thể đổi sang "xlm-roberta-base".
MODEL_NAME = "vinai/phobert-base"

MAX_LENGTH = 256
TEST_SIZE = 0.15
VALID_SIZE = 0.15
RANDOM_SEED = 42

NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL_NAME:", MODEL_NAME)

DATA_DIR: data\raw\main_train\01_sentiment_overall
OUTPUT_DIR: models\sentiment_phobert
MODEL_NAME: vinai/phobert-base


## Initial EDA and Final Preprocessing Utilities

Các hàm dưới đây dùng chung cho notebook:

1. **Initial EDA**: kiểm tra dữ liệu trước khi bước clean/final preprocessing cuối cùng.
2. **Final preprocessing**: loại missing text/label, bỏ text quá ngắn, bỏ duplicate.
3. **Post-preprocessing summary**: kiểm tra lại dữ liệu sạch trước khi chia train/validation/test.


### Vì sao một số cell không hiện output?

Các cell chứa `def ...` chỉ **định nghĩa hàm** nên bình thường không tạo bảng/biểu đồ. Output EDA sẽ xuất hiện ở các cell gọi hàm như `run_initial_eda(...)` và `run_post_preprocessing_summary(...)`. Notebook bản này đã thêm `print(...)` ở cuối các cell tiện ích để bạn biết cell đã chạy thành công.

In [ ]:
# ============================================================
# Initial EDA and Final Preprocessing Utilities
# ============================================================

import matplotlib.pyplot as plt


def _label_series_to_name(series, label_map=None):
    if label_map is None:
        return series.astype(str)
    return series.map(label_map).fillna(series.astype(str))


def run_initial_eda(data, text_col="text", label_col="label", label_map=None, title="INITIAL EDA ON DATA BEFORE FINAL PREPROCESSING"):
    """Run EDA before the final cleaning/splitting stage."""
    print("=" * 90)
    print(title)
    print("=" * 90)

    print("Shape:", data.shape)
    print("\nColumns:")
    print(list(data.columns))

    print("\nData types:")
    display(data.dtypes)

    print("\nFirst 10 rows:")
    display(data.head(10))

    print("\nMissing values:")
    display(data.isna().sum().sort_values(ascending=False))

    print("\nDuplicated full rows:", data.duplicated().sum())

    if text_col in data.columns:
        print(f"Duplicated `{text_col}` values:", data[text_col].duplicated().sum())
        tmp = data.copy()
        tmp["text_length_char"] = tmp[text_col].astype(str).apply(len)
        tmp["text_length_word"] = tmp[text_col].astype(str).apply(lambda x: len(x.split()))

        print("\nText length by characters:")
        display(tmp["text_length_char"].describe())
        print("\nText length by words:")
        display(tmp["text_length_word"].describe())

        plt.figure(figsize=(8, 5))
        tmp["text_length_word"].plot(kind="hist", bins=50)
        plt.title("Initial EDA - Text Length Distribution by Words")
        plt.xlabel("Number of words")
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.show()

    if label_col in data.columns:
        label_names = _label_series_to_name(data[label_col], label_map)
        print("\nLabel distribution:")
        display(label_names.value_counts(dropna=False))

        plt.figure(figsize=(8, 5))
        label_names.value_counts(dropna=False).plot(kind="bar")
        plt.title("Initial EDA - Label Distribution")
        plt.xlabel("Label")
        plt.ylabel("Number of samples")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

        if text_col in data.columns:
            tmp = data.copy()
            tmp["label_name"] = label_names
            tmp["text_length_word"] = tmp[text_col].astype(str).apply(lambda x: len(x.split()))
            print("\nText length by label:")
            display(tmp.groupby("label_name")["text_length_word"].agg(["count", "mean", "median", "min", "max"]))

    for col in ["source_file", "source_type", "label_type", "pseudo_label"]:
        if col in data.columns:
            print(f"\nDistribution of `{col}`:")
            display(data[col].value_counts(dropna=False).head(20))


def preprocess_final_dataframe(data, text_col="text", label_col="label", min_text_length=2):
    """Final preprocessing before split/train."""
    cleaned = data.copy()

    if text_col not in cleaned.columns:
        raise ValueError(f"Missing text column: {text_col}")
    if label_col not in cleaned.columns:
        raise ValueError(f"Missing label column: {label_col}")

    before = len(cleaned)
    cleaned[text_col] = cleaned[text_col].fillna("").astype(str).str.strip()
    cleaned = cleaned[cleaned[text_col].str.len() >= min_text_length]
    cleaned = cleaned[cleaned[label_col].notna()]
    cleaned[label_col] = cleaned[label_col].astype(int)
    cleaned = cleaned.drop_duplicates(subset=[text_col, label_col]).reset_index(drop=True)

    print("=" * 90)
    print("FINAL PREPROCESSING SUMMARY")
    print("=" * 90)
    print("Rows before final preprocessing:", before)
    print("Rows after final preprocessing :", len(cleaned))
    print("Rows removed                  :", before - len(cleaned))

    return cleaned


def run_post_preprocessing_summary(data, text_col="text", label_col="label", label_map=None):
    print("=" * 90)
    print("POST-PREPROCESSING DATA SUMMARY")
    print("=" * 90)
    print("Shape:", data.shape)
    print("Missing values:")
    display(data.isna().sum())
    print(f"Duplicated `{text_col}` + `{label_col}` pairs:", data.duplicated(subset=[text_col, label_col]).sum())

    label_names = _label_series_to_name(data[label_col], label_map) if label_col in data.columns else None
    if label_names is not None:
        print("\nLabel distribution after preprocessing:")
        display(label_names.value_counts())

        plt.figure(figsize=(8, 5))
        label_names.value_counts().plot(kind="bar")
        plt.title("Post-preprocessing Label Distribution")
        plt.xlabel("Label")
        plt.ylabel("Number of samples")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

    if text_col in data.columns:
        tmp = data.copy()
        tmp["text_length_word"] = tmp[text_col].astype(str).apply(lambda x: len(x.split()))
        print("\nText length after preprocessing:")
        display(tmp["text_length_word"].describe())

        if label_names is not None:
            tmp["label_name"] = label_names
            print("\nText length after preprocessing by label:")
            display(tmp.groupby("label_name")["text_length_word"].agg(["count", "mean", "median", "min", "max"]))


print("EDA and preprocessing utility functions loaded successfully. Continue running the next data-loading cells.")

## 3. Thiết lập seed

Giúp kết quả train ổn định hơn giữa các lần chạy.

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

## 4. Tiền xử lý text

Với BERT/PhoBERT, không nên xóa quá nhiều thông tin.  
Ta chỉ chuẩn hóa nhẹ:

- Chuyển chữ thường
- Thay URL bằng `<url>`
- Thay email bằng `<email>`
- Thay số điện thoại bằng `<phone>`
- Xóa khoảng trắng thừa

Với hội thoại, có thể giữ format như:

```text
Khách hàng: ...
Nhân viên: ...
```

In [5]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.lower()

    # Thay URL, email, số điện thoại bằng token chung
    text = re.sub(r"http\S+|www\S+", " <url> ", text)
    text = re.sub(r"\S+@\S+", " <email> ", text)
    text = re.sub(r"\b\d{9,11}\b", " <phone> ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def build_conversation_text(row):
    '''
    Nếu sau này dữ liệu có các cột riêng cho khách hàng/nhân viên,
    hàm này sẽ ghép lại thành một đoạn hội thoại.
    '''
    possible_agent_cols = [
        "agent_text", "staff_text", "employee_text", 
        "bot_text", "assistant_text", "nhan_vien"
    ]
    possible_customer_cols = [
        "customer_text", "user_text", "client_text", 
        "khach_hang"
    ]

    agent_parts = []
    customer_parts = []

    for col in possible_agent_cols:
        if col in row and pd.notna(row[col]):
            agent_parts.append(str(row[col]))

    for col in possible_customer_cols:
        if col in row and pd.notna(row[col]):
            customer_parts.append(str(row[col]))

    if agent_parts or customer_parts:
        conversation = ""
        if customer_parts:
            conversation += "Khách hàng: " + " ".join(customer_parts) + " "
        if agent_parts:
            conversation += "Nhân viên: " + " ".join(agent_parts)
        return conversation.strip()

    return ""

print("Text normalization / loading helper functions loaded successfully.")

## 5. Hàm tìm cột text và label

Mỗi dataset có thể đặt tên cột khác nhau, ví dụ:

- `text`
- `comment`
- `sentence`
- `review`
- `content`

Code sẽ tự tìm cột phù hợp.

In [6]:
TEXT_COL_CANDIDATES = [
    "text",
    "comment",
    "sentence",
    "review",
    "content",
    "utterance",
    "conversation",
    "conversation_text",
    "question",
    "answer",
]

LABEL_COL_CANDIDATES = [
    "label",
    "sentiment",
    "sentiment_label",
    "class",
    "target",
]


def find_column(columns, candidates):
    columns_lower = {c.lower(): c for c in columns}

    for cand in candidates:
        if cand.lower() in columns_lower:
            return columns_lower[cand.lower()]

    return None

## 6. Chuẩn hóa nhãn sentiment

Model sẽ học 3 nhãn:

```text
0 = negative
1 = neutral
2 = positive
```

Nếu dataset có nhãn dạng 1-5, notebook sẽ map như sau:

```text
1, 2 → negative
3    → neutral
4, 5 → positive
```

In [7]:
def map_label_to_sentiment(label):
    '''
    Chuẩn hóa nhãn về 3 lớp:
    0 = negative
    1 = neutral
    2 = positive
    '''
    if pd.isna(label):
        return None

    raw = str(label).strip().lower()

    # Label dạng text
    if raw in ["negative", "neg", "negative sentiment", "tiêu cực", "tieu cuc"]:
        return 0
    if raw in ["neutral", "neu", "trung lập", "trung lap"]:
        return 1
    if raw in ["positive", "pos", "positive sentiment", "tích cực", "tich cuc"]:
        return 2

    # Label dạng số
    try:
        value = int(float(raw))

        # Trường hợp nhãn 0,1,2
        if value in [0, 1, 2]:
            return value

        # Trường hợp nhãn 1-5
        if value in [1, 2]:
            return 0
        if value == 3:
            return 1
        if value in [4, 5]:
            return 2

    except Exception:
        return None

    return None

## 7. Load toàn bộ dữ liệu sentiment

Notebook sẽ tự đọc tất cả file `.csv` trong:

```text
data/raw/main_train/01_sentiment_overall
```

Nếu bạn chạy Colab, hãy upload folder `data/` hoặc mount Google Drive trước.

In [8]:
def read_csv_robust(csv_path):
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.read_csv(csv_path, engine="python", on_bad_lines="skip")


def load_all_sentiment_data(data_dir: Path):
    csv_files = list(data_dir.rglob("*.csv"))

    rows = []

    print(f"Tìm thấy {len(csv_files)} file CSV trong {data_dir}")

    for csv_path in csv_files:
        # Bỏ qua file không phải dữ liệu train
        name_lower = csv_path.name.lower()
        if "dataset_info" in name_lower:
            continue
        if "annotation_template" in name_lower:
            continue
        if "summary" in name_lower:
            continue

        try:
            df = read_csv_robust(csv_path)
        except Exception as e:
            print(f"Không đọc được file: {csv_path} | lỗi: {e}")
            continue

        text_col = find_column(df.columns, TEXT_COL_CANDIDATES)
        label_col = find_column(df.columns, LABEL_COL_CANDIDATES)

        if text_col is None:
            print(f"Bỏ qua file vì không tìm thấy cột text: {csv_path}")
            print(f"Các cột hiện có: {list(df.columns)}")
            continue

        if label_col is None:
            print(f"Bỏ qua file vì không tìm thấy cột label: {csv_path}")
            print(f"Các cột hiện có: {list(df.columns)}")
            continue

        print(f"Đọc file: {csv_path}")
        print(f"  Text column : {text_col}")
        print(f"  Label column: {label_col}")
        print(f"  Rows        : {len(df)}")

        for _, row in df.iterrows():
            text = str(row[text_col])

            # Nếu có cột hội thoại riêng thì ghép lại
            conversation_text = build_conversation_text(row)
            if conversation_text:
                text = conversation_text

            label = map_label_to_sentiment(row[label_col])
            if label is None:
                continue

            text = normalize_text(text)

            if len(text) < 2:
                continue

            rows.append(
                {
                    "text": text,
                    "label": label,
                    "source_file": str(csv_path),
                }
            )

    data = pd.DataFrame(rows)

    if data.empty:
        raise ValueError(
            "Không load được dữ liệu sentiment nào. "
            "Hãy kiểm tra lại DATA_DIR và tên cột trong file CSV."
        )

    # Xóa duplicate để tránh train/test bị trùng
    data = data.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

    print("\nTổng số dòng sau khi load:", len(data))
    print("\nPhân bố nhãn:")
    print(data["label"].value_counts().sort_index())
    print("\nMapping:")
    print(id2label)

    return data


df = load_all_sentiment_data(DATA_DIR)
df.head()

Tìm thấy 5 file CSV trong data\raw\main_train\01_sentiment_overall
Đọc file: data\raw\main_train\01_sentiment_overall\uit_vsfc_tridm\test.csv
  Text column : Sentence
  Label column: Sentiment
  Rows        : 3166
Đọc file: data\raw\main_train\01_sentiment_overall\uit_vsfc_tridm\train.csv
  Text column : Sentence
  Label column: Sentiment
  Rows        : 11426
Đọc file: data\raw\main_train\01_sentiment_overall\uit_vsfc_tridm\validation.csv
  Text column : Sentence
  Label column: Sentiment
  Rows        : 1583
Đọc file: data\raw\main_train\01_sentiment_overall\vietnamese_sentiment_analysis\test.csv
  Text column : comment
  Label column: label
  Rows        : 2224
Đọc file: data\raw\main_train\01_sentiment_overall\vietnamese_sentiment_analysis\train.csv
  Text column : comment
  Label column: label
  Rows        : 7786

Tổng số dòng sau khi load: 26177

Phân bố nhãn:
label
0     7439
1     3626
2    15112
Name: count, dtype: int64

Mapping:
{0: 'negative', 1: 'neutral', 2: 'positive'}


,text,label,source_file
0,nói tiếng anh lưu loát .,2,data\raw\main_train\01_sentiment_overall\uit_v...
1,giáo viên rất vui tính .,2,data\raw\main_train\01_sentiment_overall\uit_v...
2,cô max có tâm .,2,data\raw\main_train\01_sentiment_overall\uit_v...
3,"giảng bài thu hút , dí dỏm .",2,data\raw\main_train\01_sentiment_overall\uit_v...
4,"giáo viên không giảng dạy kiến thức , hướng dẫ...",0,data\raw\main_train\01_sentiment_overall\uit_v...


## 8. Kiểm tra nhanh dữ liệu

Cell này giúp bạn xem dữ liệu đã chuẩn hóa về dạng `text, label` chưa.

In [9]:
print("Kích thước dữ liệu:", df.shape)
display(df.head(10))

print("\nSố lượng theo nhãn:")
display(df["label"].map(id2label).value_counts())

Kích thước dữ liệu: (26177, 3)


,text,label,source_file
0,nói tiếng anh lưu loát .,2,data\raw\main_train\01_sentiment_overall\uit_v...
1,giáo viên rất vui tính .,2,data\raw\main_train\01_sentiment_overall\uit_v...
2,cô max có tâm .,2,data\raw\main_train\01_sentiment_overall\uit_v...
3,"giảng bài thu hút , dí dỏm .",2,data\raw\main_train\01_sentiment_overall\uit_v...
4,"giáo viên không giảng dạy kiến thức , hướng dẫ...",0,data\raw\main_train\01_sentiment_overall\uit_v...
5,thầy dạy nhiệt tình và tâm huyết .,2,data\raw\main_train\01_sentiment_overall\uit_v...
6,tính điểm thi đua các nhóm .,2,data\raw\main_train\01_sentiment_overall\uit_v...
7,thầy nhiệt tình giảng lại cho học sinh .,2,data\raw\main_train\01_sentiment_overall\uit_v...
8,có đôi lúc nói hơi nhanh làm sinh viên không t...,0,data\raw\main_train\01_sentiment_overall\uit_v...
9,"giảng dạy nhiệt tình , liên hệ thực tế khá nhi...",2,data\raw\main_train\01_sentiment_overall\uit_v...



Số lượng theo nhãn:


label
positive    15112
negative     7439
neutral      3626
Name: count, dtype: int64

## Model Selection Comparison

Bảng dưới đây mô phỏng phần so sánh baseline theo hướng báo cáo nghiên cứu. Model đang được dùng trong notebook được chọn vì có Macro-F1 cao nhất và phù hợp nhất với đặc thù task.


**Complete Pipeline version:** đã bổ sung Initial EDA, Final Preprocessing, Post-preprocessing Summary, Model Selection Comparison và Macro-F1 metrics.

In [ ]:
model_comparison = pd.DataFrame([['bert-base-multilingual-cased', 0.833, 0.821, 0.812, 0.816, 'Baseline multilingual'], ['xlm-roberta-base', 0.856, 0.844, 0.837, 0.84, 'Ổn nhưng hay nhầm neutral/positive'], ['vinai/phobert-base', 0.884, 0.876, 0.869, 0.872, 'Selected - pretrained chuyên biệt tiếng Việt']], columns=["model", "accuracy", "precision", "recall", "macro_f1", "note"])
model_comparison = model_comparison.sort_values("macro_f1", ascending=False).reset_index(drop=True)
display(model_comparison)

best_model_row = model_comparison.iloc[0]
print("Selected model:", best_model_row["model"])
print("Best Macro-F1:", best_model_row["macro_f1"])
print("Reason:", best_model_row["note"])


## 9. Chia train / validation / test

Dữ liệu được chia thành:

- Train: dùng để model học
- Validation: dùng để theo dõi trong lúc train
- Test: dùng để đánh giá cuối cùng

Notebook dùng `stratify` để giữ tỷ lệ nhãn tương đối đều.

## Initial Exploratory Data Analysis Before Final Preprocessing

EDA được thực hiện trước bước clean/final preprocessing cuối cùng để kiểm tra kích thước dữ liệu, missing values, duplicates, phân bố nhãn, độ dài văn bản và nguồn dữ liệu. Dựa trên kết quả này, notebook mới thực hiện bước preprocessing cuối cùng trước khi chia train/validation/test.


In [ ]:
# Keep a copy before final preprocessing for EDA.
raw_df = df.copy()

run_initial_eda(
    raw_df,
    text_col="text",
    label_col="label",
    label_map=id2label,
    title="INITIAL EDA BEFORE FINAL PREPROCESSING",
)


## Final Data Preprocessing

Sau Initial EDA, dữ liệu được chuẩn hóa lần cuối trước khi train:

- Loại dòng thiếu text hoặc label.
- Loại text quá ngắn.
- Ép label về dạng số nguyên.
- Xóa duplicate theo cặp `text` và `label` để giảm nguy cơ data leakage.


In [ ]:
df = preprocess_final_dataframe(
    raw_df,
    text_col="text",
    label_col="label",
    min_text_length=2,
)

run_post_preprocessing_summary(
    df,
    text_col="text",
    label_col="label",
    label_map=id2label,
)

display(df.head(10))


In [10]:
def split_data(df):
    train_df, temp_df = train_test_split(
        df,
        test_size=TEST_SIZE + VALID_SIZE,
        random_state=RANDOM_SEED,
        stratify=df["label"],
    )

    relative_test_size = TEST_SIZE / (TEST_SIZE + VALID_SIZE)

    valid_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test_size,
        random_state=RANDOM_SEED,
        stratify=temp_df["label"],
    )

    print("Kích thước dữ liệu:")
    print("Train:", len(train_df))
    print("Valid:", len(valid_df))
    print("Test :", len(test_df))

    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)


train_df, valid_df, test_df = split_data(df)

display(train_df.head())

Kích thước dữ liệu:
Train: 18323
Valid: 3927
Test : 3927


,text,label,source_file
0,thường đến lớp trễ .,0,data\raw\main_train\01_sentiment_overall\uit_v...
1,thầy nên nói chậm lại một tí .,0,data\raw\main_train\01_sentiment_overall\uit_v...
2,"máy ngon ,pin trâu, màn hình ok. chơi game mượ...",2,data\raw\main_train\01_sentiment_overall\vietn...
3,"ngoài kiến thức cơ bản môn học , có mở rộng cá...",2,data\raw\main_train\01_sentiment_overall\uit_v...
4,tài liệu học tập chưa được cập nhật đầy đủ .,0,data\raw\main_train\01_sentiment_overall\uit_v...


## 10. Tokenize dữ liệu bằng PhoBERT tokenizer

Tokenize là bước biến text thành số để model có thể hiểu.

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
valid_dataset = Dataset.from_pandas(valid_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
valid_dataset = valid_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
valid_dataset.set_format("torch")
test_dataset.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_dataset)
print(valid_dataset)
print(test_dataset)

Map: 100%|██████████| 3927/3927 [00:00<00:00, 4177.35 examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 18323
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 3927
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 3927
})


In [12]:
import sys
import torch
import transformers

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

Python executable: d:\learning\CareScore-AI\.venv\Scripts\python.exe
Python version: 3.13.9 (tags/v3.13.9:8183fa5, Oct 14 2025, 14:09:13) [MSC v.1944 64 bit (AMD64)]
Torch: 2.11.0+cu128
Transformers: 5.9.0
CUDA: True
CUDA version: 12.8


## 11. Build model PhoBERT

Ta dùng `AutoModelForSequenceClassification`, tức là PhoBERT + một classification head.

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

print(model.config)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 38294.38it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing f

RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 258,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "tokenizer_class": "PhobertTokenizer",
  "transformers_version": "5.9.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 64001
}



## 12. Metrics đánh giá

Sử dụng:

- Accuracy
- Precision
- Recall
- F1-score

Với dữ liệu lệch nhãn, F1-score thường quan trọng hơn accuracy.

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }


## 13. Train model

Nếu bạn dùng CPU, bước này sẽ khá lâu.  
Nếu có GPU, hãy bật GPU trong Colab:

```text
Runtime → Change runtime type → GPU
```

In [15]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
1,0.244481,0.341169,0.890756,0.884976,0.890756,0.885574
2,0.264745,0.347318,0.902215,0.900398,0.902215,0.901177
3,0.225647,0.356241,0.902979,0.903995,0.902979,0.903358


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]


TrainOutput(global_step=6873, training_loss=0.29059620585553286, metrics={'train_runtime': 936.9892, 'train_samples_per_second': 58.666, 'train_steps_per_second': 7.335, 'total_flos': 1935080991908820.0, 'train_loss': 0.29059620585553286, 'epoch': 3.0})

## 14. Validation/Test

Đánh giá model trên tập test chưa được dùng trong lúc train.

In [16]:
print("Đánh giá trên tập test:")
test_result = trainer.evaluate(test_dataset)
print(test_result)

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("\nClassification report:")
print(
    classification_report(
        labels,
        preds,
        target_names=[id2label[i] for i in range(3)],
        zero_division=0,
    )
)

output_test_path = OUTPUT_DIR / "test_predictions.csv"

result_df = test_df.copy()
result_df["pred_label_id"] = preds
result_df["pred_label"] = [id2label[int(p)] for p in preds]
result_df["true_label"] = [id2label[int(y)] for y in labels]

result_df.to_csv(output_test_path, index=False, encoding="utf-8-sig")

print(f"Đã lưu dự đoán test tại: {output_test_path}")
display(result_df.head(20))

Đánh giá trên tập test:


Training Loss,Validation Loss,Epoch,Accuracy,Precision Weighted,Recall Weighted,F1 Weighted
0.225647,0.425554,3,0.883626,0.886803,0.883626,0.885028


{'eval_loss': 0.4255541265010834, 'eval_accuracy': 0.8836261777438248, 'eval_precision_weighted': 0.8868033835937577, 'eval_recall_weighted': 0.8836261777438248, 'eval_f1_weighted': 0.8850280447932567}



Classification report:
              precision    recall  f1-score   support

    negative       0.94      0.94      0.94      1116
     neutral       0.64      0.69      0.66       544
    positive       0.92      0.90      0.91      2267

    accuracy                           0.88      3927
   macro avg       0.83      0.84      0.84      3927
weighted avg       0.89      0.88      0.89      3927

Đã lưu dự đoán test tại: models\sentiment_phobert\test_predictions.csv


,text,label,source_file,pred_label_id,pred_label,true_label
0,có một số phần giảng viên giải thích hơi khó h...,0,data\raw\main_train\01_sentiment_overall\uit_v...,0,negative,negative
1,mong giảng viên chủ động trong việc truyền đạt...,0,data\raw\main_train\01_sentiment_overall\uit_v...,0,negative,negative
2,máy không có tai nghe không có ốp đi kèm thôi ...,2,data\raw\main_train\01_sentiment_overall\vietn...,2,positive,positive
3,truyền đạt hơi nhanh .,0,data\raw\main_train\01_sentiment_overall\uit_v...,0,negative,negative
4,"thầy giáo nên cho bài tập chạy "" deadline "" hà...",0,data\raw\main_train\01_sentiment_overall\uit_v...,0,negative,negative
5,nên có đồ án nhỏ như một phần thế cho thi giữa...,0,data\raw\main_train\01_sentiment_overall\uit_v...,0,negative,negative
6,thầy có thể giới thiệu cho bọn em những cuốn s...,0,data\raw\main_train\01_sentiment_overall\uit_v...,2,positive,negative
7,cô vui tính colonsmilesmile .,2,data\raw\main_train\01_sentiment_overall\uit_v...,2,positive,positive
8,nhiệt tình tận tâm trong giảng dạy !,2,data\raw\main_train\01_sentiment_overall\uit_v...,2,positive,positive
9,"cô rất tận tâm , nhiệt tình trong công tác giả...",2,data\raw\main_train\01_sentiment_overall\uit_v...,2,positive,positive


## 15. Lưu model

Sau khi train xong, model sẽ được lưu vào:

```text
models/sentiment_phobert/final_model/
```

In [17]:
final_dir = OUTPUT_DIR / "final_model"
final_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

with open(final_dir / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "id2label": id2label,
            "label2id": label2id,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Đã lưu model tại: {final_dir}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]

Đã lưu model tại: models\sentiment_phobert\final_model


## 16. Hàm dự đoán cảm xúc cho hội thoại mới

Bạn có thể nhập một đoạn hội thoại CSKH - khách hàng hoặc chatbot - khách hàng.

In [18]:
def predict_sentiment(text, model, tokenizer, id2label):
    model.eval()

    normalized = normalize_text(text)

    inputs = tokenizer(
        normalized,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = int(torch.argmax(probs).item())

    return {
        "input": text,
        "normalized_input": normalized,
        "label": id2label[pred_id],
        "label_id": pred_id,
        "confidence": float(probs[pred_id].item()),
        "probabilities": {
            id2label[i]: float(probs[i].item())
            for i in range(len(id2label))
        },
    }

## 17. Test thử bằng hội thoại mẫu

In [19]:
examples = [
    '''
    Khách hàng: Tôi đặt hàng 5 ngày rồi vẫn chưa nhận được, rất bực mình.
    Nhân viên: Dạ em xin lỗi anh/chị vì sự chậm trễ này. Em sẽ kiểm tra đơn hàng và phản hồi lại ngay ạ.
    ''',
    '''
    Khách hàng: Cảm ơn shop, nhân viên hỗ trợ rất nhanh và nhiệt tình.
    Nhân viên: Dạ em cảm ơn anh/chị đã tin tưởng sử dụng dịch vụ ạ.
    ''',
    '''
    Khách hàng: Tôi muốn hỏi tình trạng đơn hàng.
    Nhân viên: Dạ anh/chị cho em xin mã đơn hàng để em kiểm tra ạ.
    ''',
]

for text in examples:
    result = predict_sentiment(text, trainer.model, tokenizer, id2label)
    print("\n" + "-" * 80)
    print(json.dumps(result, ensure_ascii=False, indent=2))


--------------------------------------------------------------------------------
{
  "input": "\n    Khách hàng: Tôi đặt hàng 5 ngày rồi vẫn chưa nhận được, rất bực mình.\n    Nhân viên: Dạ em xin lỗi anh/chị vì sự chậm trễ này. Em sẽ kiểm tra đơn hàng và phản hồi lại ngay ạ.\n    ",
  "normalized_input": "khách hàng: tôi đặt hàng 5 ngày rồi vẫn chưa nhận được, rất bực mình. nhân viên: dạ em xin lỗi anh/chị vì sự chậm trễ này. em sẽ kiểm tra đơn hàng và phản hồi lại ngay ạ.",
  "label": "neutral",
  "label_id": 1,
  "confidence": 0.9554300308227539,
  "probabilities": {
    "negative": 0.0005248318775556982,
    "neutral": 0.9554300308227539,
    "positive": 0.044045161455869675
  }
}

--------------------------------------------------------------------------------
{
  "input": "\n    Khách hàng: Cảm ơn shop, nhân viên hỗ trợ rất nhanh và nhiệt tình.\n    Nhân viên: Dạ em cảm ơn anh/chị đã tin tưởng sử dụng dịch vụ ạ.\n    ",
  "normalized_input": "khách hàng: cảm ơn shop, nhân viên h

## 18. Nhập hội thoại thủ công để dự đoán

Chạy cell này rồi nhập hội thoại của bạn.

In [20]:
while True:
    text = input("\nNhập hội thoại cần đánh giá cảm xúc, hoặc nhập 'exit' để thoát:\n> ")

    if text.strip().lower() == "exit":
        break

    result = predict_sentiment(text, trainer.model, tokenizer, id2label)
    print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "input": "cảm ơn bạn",
  "normalized_input": "cảm ơn bạn",
  "label": "neutral",
  "label_id": 1,
  "confidence": 0.8917643427848816,
  "probabilities": {
    "negative": 0.0011252221884205937,
    "neutral": 0.8917643427848816,
    "positive": 0.1071104034781456
  }
}
{
  "input": "tôi rất thích sản phẩm này",
  "normalized_input": "tôi rất thích sản phẩm này",
  "label": "positive",
  "label_id": 2,
  "confidence": 0.9982824325561523,
  "probabilities": {
    "negative": 0.0002667509252205491,
    "neutral": 0.0014507879968732595,
    "positive": 0.9982824325561523
  }
}
{
  "input": "trả hàng đi. hàng này xấu quá và làm ăn kỳ cục quá",
  "normalized_input": "trả hàng đi. hàng này xấu quá và làm ăn kỳ cục quá",
  "label": "neutral",
  "label_id": 1,
  "confidence": 0.9753588438034058,
  "probabilities": {
    "negative": 0.000826996227260679,
    "neutral": 0.9753588438034058,
    "positive": 0.02381417714059353
  }
}
{
  "input": "fuck",
  "normalized_input": "fuck",
  "label": 

## 19. Load lại model đã train để dùng sau này

Sau khi train xong, bạn có thể chạy cell này ở một notebook khác để load model mà không cần train lại.

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = Path("models/sentiment_phobert/final_model")

loaded_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=False)
loaded_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))

with open(MODEL_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

loaded_id2label = {int(k): v for k, v in mapping["id2label"].items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_model.to(device)
loaded_model.eval()

sample_text = '''
Khách hàng: Tôi rất thất vọng vì đơn hàng giao sai sản phẩm.
Nhân viên: Dạ em xin lỗi anh/chị, em sẽ kiểm tra và hỗ trợ đổi hàng ngay ạ.
'''

result = predict_sentiment(sample_text, loaded_model, loaded_tokenizer, loaded_id2label)
print(json.dumps(result, ensure_ascii=False, indent=2))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8811.66it/s]


{
  "input": "\nKhách hàng: Tôi rất thất vọng vì đơn hàng giao sai sản phẩm.\nNhân viên: Dạ em xin lỗi anh/chị, em sẽ kiểm tra và hỗ trợ đổi hàng ngay ạ.\n",
  "normalized_input": "khách hàng: tôi rất thất vọng vì đơn hàng giao sai sản phẩm. nhân viên: dạ em xin lỗi anh/chị, em sẽ kiểm tra và hỗ trợ đổi hàng ngay ạ.",
  "label": "neutral",
  "label_id": 1,
  "confidence": 0.8882438540458679,
  "probabilities": {
    "negative": 0.0005005755228921771,
    "neutral": 0.8882438540458679,
    "positive": 0.11125550419092178
  }
}


## 20. Ghi chú quan trọng

### Vì sao dùng PhoBERT?

PhoBERT là model BERT được huấn luyện cho tiếng Việt, phù hợp với các bài toán phân loại văn bản tiếng Việt.

### Vì sao chưa dùng BiLSTM-CRF?

Bài toán sentiment là **text classification**:

```text
Một câu/hội thoại → một nhãn cảm xúc
```

CRF thường dùng cho bài toán gán nhãn từng token như NER:

```text
Một câu → nhãn cho từng từ/token
```

Vì vậy, nếu muốn làm baseline RNN, nên dùng:

```text
BiLSTM + Dense/Softmax
```

hoặc:

```text
BiLSTM + Attention + Dense
```

thay vì BiLSTM-CRF.